# UMG Label Operations Telemetry Processor
**Core Data Engineering & Feature Processing Pipeline Layer**

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Create local data directory structure if not exists
os.makedirs('data', exist_ok=True)
os.makedirs('notebooks', exist_ok=True)

print("Initializing UMG Label Operations Data Simulation Engine [May 2026]...")

# 1. Setup Simulation Parameters matching explicit user data schema
np.random.seed(42)
n_rows = 1000

artists = [f"Artist_{i}" for i in range(1, 21)]
regions = ["US-Latin", "Mexico", "Brazil", "Argentina", "Spain"]
dsps = ["Spotify", "Apple Music", "YouTube Music", "Amazon Music"]
genres = ["Reggaeton", "Regional Mexican", "Rock En Español", "Latin Trap", "Pop"]

timestamps = pd.date_range(start="2026-01-01", end="2026-05-15", periods=n_rows)

data = {
    "timestamp": timestamps,
    "artist_name": np.random.choice(artists, size=n_rows),
    "region": np.random.choice(regions, size=n_rows),
    "dsp": np.random.choice(dsps, size=n_rows),
    "genre": np.random.choice(genres, size=n_rows),
    "isrc": [f"USUMG26{np.random.randint(10000, 99999)}" for _ in range(n_rows)],
    "streams": np.random.exponential(scale=5000, size=n_rows).astype(int) + 100,
    "marketing_spend_usd": np.round(np.random.gamma(shape=2, scale=200, size=n_rows), 2)
}

# Inject controlled anomalies representing hyper-localized viral breakouts
for i in range(len(data["streams"])):
    # Simulating sudden breakout events for Artist_1 and Artist_7 in LATAM regions
    if data["artist_name"][i] in ["Artist_1", "Artist_7"] and data["region"][i] in ["Mexico", "Brazil"]:
        data["streams"][i] *= np.random.randint(12, 25)
        # Often marketing spend lags behind real-time viral spikes
        if np.random.rand() > 0.6:
            data["marketing_spend_usd"][i] = 0.0

df = pd.DataFrame(data)
df.to_csv("data/umg_label_ops_sim.csv", index=False)
print("✔ Base simulated telemetry data successfully compiled to 'data/umg_label_ops_sim.csv'.")

# 2. Advanced Feature Engineering & Analytics Pipeline
print("
Processing streaming telemetry data and executing financial governance calculations...")

# Read from source to ensure end-to-end processing pipeline validity
raw_data = pd.read_csv("data/umg_label_ops_sim.csv", parse_dates=["timestamp"])

# Calculate Operational metrics: Cost per Thousand Streams (CPM) and Yield
# To prevent division by zero anomalies, ensure zero spend is handled correctly
raw_data["cpm_usd"] = np.where(
    raw_data["streams"] > 0,
    (raw_data["marketing_spend_usd"] / raw_data["streams"]) * 1000,
    0.0
)
raw_data["cpm_usd"] = np.round(raw_data["cpm_usd"], 2)

# Sort by time series sequence to properly calculate rolling performance parameters
raw_data = raw_data.sort_values(by=["artist_name", "region", "timestamp"]).reset_index(drop=True)

# Calculate a 3-period rolling average of streams to identify track velocity trajectory
raw_data["rolling_avg_streams"] = raw_data.groupby(["artist_name", "region"])["streams"]                                            .transform(lambda x: x.rolling(window=3, min_periods=1).mean())
raw_data["rolling_avg_streams"] = np.round(raw_data["rolling_avg_streams"], 1)

# Establish Financial Governance Metric: Budget Overrun Identification
# Setting a conceptual baseline threshold budget of $500 per tracked operational asset window
BUDGET_BASELINE_USD = 500.00
raw_data["budget_variance_usd"] = raw_data["marketing_spend_usd"] - BUDGET_BASELINE_USD

# Define explicit operational action triggers based on financial risk profile
raw_data["financial_status"] = np.where(
    raw_data["budget_variance_usd"] > 150.00, "CRITICAL_OVER_BUDGET",
    np.where(raw_data["marketing_spend_usd"] == 0.0, "UNDER_FUNDED_VIRAL_RISK", "OPTIMIZED")
)

# 3. Generate Analytical Portfolio Insights Summary Groupings
print("
--- EXECUTIVE OPERATIONAL INTELLIGENCE DASHBOARD DATA SUMMARY ---")
regional_summary = raw_data.groupby("region").agg(
    total_streams=("streams", "sum"),
    total_spend_usd=("marketing_spend_usd", "sum"),
    average_cpm_usd=("cpm_usd", "mean")
).reset_index()

regional_summary["roas_index"] = np.round(regional_summary["total_streams"] / regional_summary["total_spend_usd"], 2)
print(regional_summary.to_string(index=False))

# 4. Generate Production Visualization Assets
print("
Generating exploratory and programmatic data visualizations...")
plt.figure(figsize=(10, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

plt.bar(regional_summary["region"], regional_summary["average_cpm_usd"], color=colors, edgecolor='black', alpha=0.8)
plt.title("UMG Label Operations: Average Cost Per 1K Streams (CPM) by Region (May 2026)", fontsize=12, fontweight='bold')
plt.xlabel("Target Crossover Region", fontsize=10)
plt.ylabel("Average CPM (USD)", fontsize=10)
plt.grid(axis='y', linestyle='--', alpha=0.5)

# Ensure layout fits cleanly
plt.tight_layout()
plt.savefig("notebooks/regional_cpm_efficiency.png", dpi=150)
plt.close()
print("✔ Visualization asset generated successfully and saved to 'notebooks/regional_cpm_efficiency.png'.")
print("
Python Processing Pipeline Execution Complete. Data layers validated successfully.")
